# 02 — Limpeza ENEM (2012–2024)

**Objetivo:** processar os microdados brutos do ENEM de cada ano, aplicar filtros de qualidade, criar variáveis derivadas e exportar um arquivo Parquet por ano.

O processamento usa **DuckDB in-memory** para leitura dos CSVs via SQL — atendendo ao requisito da disciplina de usar SQL no pipeline. O DuckDB é muito mais eficiente que o pandas puro para ler arquivos CSV de 5–8 milhões de linhas.

**Inputs:** `datasets/enem/microdados_enem_AAAA/DADOS/MICRODADOS_ENEM_AAAA.csv`  
- Separador: `;` | Encoding: `latin-1`  
- **2016:** nome de arquivo em minúsculo (`microdados_enem_2016.csv`) — tratado automaticamente  
- **2024:** estrutura dividida em dois arquivos separados (ver bloco especial abaixo)

**Output:** `data/processed/enem/enem_AAAA.parquet` — um arquivo por ano (2012–2024)

## 1. Imports e configuração

In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

Path('../data/processed/enem').mkdir(parents=True, exist_ok=True)

ANOS = list(range(2012, 2025))


## 2. Constantes e mapeamentos

### Colunas selecionadas
Selecionamos apenas as colunas relevantes para a análise, descartando campos de gabarito, itens de prova e localização de aplicação que não são usados nos modelos.

### Harmonização do Q006 (renda familiar)
O questionário socioeconômico mudou em 2015 — a escala de renda foi reorganizada de 10 para 16 categorias. Para permitir análise temporal comparável, mapeamos ambas as escalas para 5 faixas comuns:
- **A** = sem renda
- **B** = até 1 salário mínimo  
- **C** = 1 a 2 salários mínimos  
- **D** = 2 a 5 salários mínimos  
- **E** = acima de 5 salários mínimos

### Mapeamento de macrorregiões
Derivamos a macrorregião diretamente da sigla UF da escola (`SG_UF_ESC`), que serve como proxy para o estado de residência do candidato.

### Faixas de nota
Bins: `[0, 400, 500, 600, 700, 800, 1001]` — mesmas faixas para CN, CH, LC, MT e Redação, usadas como variável alvo nos modelos de classificação.

In [ ]:
COLUNAS = [
    'NU_ANO', 'CO_MUNICIPIO_ESC', 'SG_UF_ESC',
    'TP_SEXO', 'TP_COR_RACA', 'TP_ESCOLA',
    'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT',
    'Q001', 'Q002', 'Q006',
    'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO'
]

Q006_2012_2014 = {'A':'A','B':'B','C':'C','D':'D','E':'D','F':'E','G':'E','H':'E','I':'E','J':'E'}
Q006_2015_2023 = {'A':'A','B':'B','C':'C','D':'C','E':'D','F':'D','G':'D','H':'D',
                  'I':'E','J':'E','K':'E','L':'E','M':'E','N':'E','O':'E','P':'E'}

SG_UF_REGIAO = {
    'RO':'Norte','AC':'Norte','AM':'Norte','RR':'Norte','PA':'Norte','AP':'Norte','TO':'Norte',
    'MA':'Nordeste','PI':'Nordeste','CE':'Nordeste','RN':'Nordeste','PB':'Nordeste',
    'PE':'Nordeste','AL':'Nordeste','SE':'Nordeste','BA':'Nordeste',
    'MG':'Sudeste','ES':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    'PR':'Sul','SC':'Sul','RS':'Sul',
    'MS':'Centro-Oeste','MT':'Centro-Oeste','GO':'Centro-Oeste','DF':'Centro-Oeste'
}

BINS   = [0, 400, 500, 600, 700, 800, 1001]
LABELS = ['<400','400-500','500-600','600-700','700-800','>800']
AREAS  = ['CN','CH','LC','MT','REDACAO']
COLS_STR = ['TP_SEXO','TP_COR_RACA','TP_ESCOLA','SG_UF_ESC',
            'Q001','Q002','Q006','Q006_HARM','REGIAO',
            'FAIXA_CN','FAIXA_CH','FAIXA_LC','FAIXA_MT','FAIXA_REDACAO']


## 3. Loop de processamento por ano

### Filtro de presença
Mantemos apenas candidatos que fizeram **todos os quatro dias de prova** e têm nota de Matemática válida:
```sql
WHERE TP_PRESENCA_CN = 1 AND TP_PRESENCA_CH = 1
  AND TP_PRESENCA_LC = 1 AND TP_PRESENCA_MT = 1
  AND NU_NOTA_MT IS NOT NULL
```
Isso remove em média ~33% das inscrições (55% em 2020 por conta do COVID-19).

### Tratamento especial 2024
O INEP separou os microdados de 2024 em dois arquivos:
- `PARTICIPANTES_2024.csv`: dados demográficos e questionário (Q006 etc.)
- `RESULTADOS_2024.csv`: escola, presença e notas

Como não há chave de join explícita entre eles, o join é feito por **posição de linha** via `ROW_NUMBER() OVER ()` — os dois arquivos têm exatamente 4.332.944 linhas na mesma ordem de inscrição.

O `TP_ESCOLA` de 2024 é derivado de `TP_DEPENDENCIA_ADM_ESC`:  
`4 (privada) → 3` | `demais (federal/estadual/municipal) → 2`

### Padronização de tipos
Todas as colunas categóricas são forçadas para `str` antes de salvar — necessário porque `TP_SEXO` era inteiro (`0`/`1`) em 2012–2014 e string (`'M'`/`'F'`) de 2015 em diante. Sem isso, o `read_parquet(..., union_by_name=true)` no notebook 04 falharia com erro de tipo.

In [ ]:
def get_csv_path(ano):
    base = f'../datasets/enem/microdados_enem_{ano}/DADOS/'
    upper = f'{base}MICRODADOS_ENEM_{ano}.csv'
    lower = f'{base}microdados_enem_{ano}.csv'
    return upper if Path(upper).exists() else lower

con = duckdb.connect()  # in-memory
resumo = []

for ano in ANOS:
    if ano == 2024:
        base   = '../datasets/enem/microdados_enem_2024/DADOS/'
        p_path = f'{base}PARTICIPANTES_2024.csv'
        r_path = f'{base}RESULTADOS_2024.csv'
        df = con.execute(f"""
            WITH p AS (
                SELECT ROW_NUMBER() OVER () AS rn,
                       NU_ANO, TP_SEXO, TP_COR_RACA, Q001, Q002, Q006
                FROM read_csv_auto('{p_path}', delim=';', header=true, ignore_errors=true)
            ),
            r AS (
                SELECT ROW_NUMBER() OVER () AS rn,
                       CO_MUNICIPIO_ESC, SG_UF_ESC, TP_DEPENDENCIA_ADM_ESC,
                       TP_PRESENCA_CN, TP_PRESENCA_CH, TP_PRESENCA_LC, TP_PRESENCA_MT,
                       NU_NOTA_CN, NU_NOTA_CH, NU_NOTA_LC, NU_NOTA_MT, NU_NOTA_REDACAO
                FROM read_csv_auto('{r_path}', delim=';', header=true, ignore_errors=true)
            )
            SELECT p.NU_ANO, r.CO_MUNICIPIO_ESC, r.SG_UF_ESC,
                   p.TP_SEXO, p.TP_COR_RACA,
                   CASE r.TP_DEPENDENCIA_ADM_ESC WHEN 4 THEN 3 ELSE 2 END AS TP_ESCOLA,
                   r.TP_PRESENCA_CN, r.TP_PRESENCA_CH, r.TP_PRESENCA_LC, r.TP_PRESENCA_MT,
                   p.Q001, p.Q002, p.Q006,
                   r.NU_NOTA_CN, r.NU_NOTA_CH, r.NU_NOTA_LC, r.NU_NOTA_MT, r.NU_NOTA_REDACAO
            FROM p JOIN r ON p.rn = r.rn
            WHERE r.TP_PRESENCA_CN=1 AND r.TP_PRESENCA_CH=1
              AND r.TP_PRESENCA_LC=1 AND r.TP_PRESENCA_MT=1
              AND r.NU_NOTA_MT IS NOT NULL
        """).df()
        total_antes = con.execute(
            f"SELECT COUNT(*) FROM read_csv_auto('{r_path}', delim=';', header=true, ignore_errors=true)"
        ).fetchone()[0]
    else:
        csv = get_csv_path(ano)
        cols_sql = ', '.join(COLUNAS)
        df = con.execute(f"""
            SELECT {cols_sql}
            FROM read_csv_auto('{csv}', delim=';', header=true, ignore_errors=true)
            WHERE TP_PRESENCA_CN=1 AND TP_PRESENCA_CH=1
              AND TP_PRESENCA_LC=1 AND TP_PRESENCA_MT=1
              AND NU_NOTA_MT IS NOT NULL
        """).df()
        total_antes = con.execute(
            f"SELECT COUNT(*) FROM read_csv_auto('{csv}', delim=';', header=true, ignore_errors=true)"
        ).fetchone()[0]

    for area in AREAS:
        df[f'FAIXA_{area}'] = pd.cut(df[f'NU_NOTA_{area}'], bins=BINS, labels=LABELS, right=False)

    mapa = Q006_2012_2014 if ano <= 2014 else Q006_2015_2023
    df['Q006_HARM'] = df['Q006'].map(mapa)
    df['REGIAO']    = df['SG_UF_ESC'].map(SG_UF_REGIAO)

    for col in COLS_STR:
        if col in df.columns:
            df[col] = df[col].astype(str).where(df[col].notna(), None)

    df.to_parquet(f'../data/processed/enem/enem_{ano}.parquet', index=False)
    resumo.append({'ano': ano, 'total_bruto': total_antes,
                   'apos_filtro': len(df), 'removidos': total_antes - len(df)})
    print(f'{ano}: {total_antes:,} -> {len(df):,} ({total_antes-len(df):,} removidos)')

con.close()
pd.DataFrame(resumo)


## 4. Verificação Q4 — Candidatos por ano (query SQL obrigatória)

Lê todos os parquets gerados com DuckDB in-memory para verificar a contagem por ano após o filtro de presença.  
Esta é a **Query Q4** exigida pela disciplina.

In [ ]:
con2 = duckdb.connect()
print(con2.execute("""
    SELECT NU_ANO, COUNT(*) AS candidatos
    FROM read_parquet('../data/processed/enem/enem_*.parquet', union_by_name=true)
    GROUP BY NU_ANO ORDER BY NU_ANO
""").df().to_string(index=False))
con2.close()
